In [1]:
!apt-get install openjdk-17-jdk-headless -qq > /dev/null
!wget -q https://downloads.apache.org/spark/spark-4.1.2/spark-4.1.2-bin-hadoop3.tgz
!tar xf spark-4.1.2-bin-hadoop3.tgz
!pip install -q findspark
!pip install -q pyspark


import findspark

import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()
sc = spark.sparkContext

## Transformaciones

In [21]:
# map()
rdd = sc.parallelize([1,2,3,4])
rdd_map = rdd.map(lambda x: x**2) # eleva al cuadrado
rdd_map.collect()

[1, 4, 9, 16]

In [5]:
# filter()
rdd = sc.parallelize([1,2,3,4])
rdd_filter = rdd.filter(lambda x: x%2 == 0) # solamente los números pares
rdd_filter.collect()

[2, 4]

In [7]:
# flatMap()
rdd = sc.parallelize(["Hola Mundo", "PySpark Spark"])
rdd_flatmap = rdd.flatMap(lambda x: x.split(" "))
rdd_flatmap.collect()

['Hola', 'Mundo', 'PySpark', 'Spark']

In [11]:
# mapPartitions(func)
rdd = sc.parallelize([1,2,3,4, 5, 6], 2)
def suma_particion(iterator):
  yield sum(iterator)
rdd_particion = rdd.mapPartitions(suma_particion)
rdd_particion.collect()

[6, 15]

In [12]:
# mapPartitionsWithIndex(func)
rdd = sc.parallelize([1,2,3,4, 5, 6], 2)
def mostrar_indice(index, iterator):
  yield f"Partición: {index}, Datos: {list(iterator)}"
rdd_index = rdd.mapPartitionsWithIndex(mostrar_indice)
rdd_index.collect()

['Partición: 0, Datos: [1, 2, 3]', 'Partición: 1, Datos: [4, 5, 6]']

In [13]:
# sample(withReplacement, fraction, seed)
rdd = sc.parallelize(range(100))
rdd_muestra = rdd.sample(False, 0.1,42) # muestra aproximada del 10% con semilla al 42
rdd_muestra.collect()

[0, 3, 10, 17, 18, 32, 35, 56, 67, 70, 73, 79, 88, 94]

In [14]:
# union
rdd1 = sc.parallelize([1,2])
rdd2 = sc.parallelize([3,4])
rdd_union = rdd1.union(rdd2)
rdd_union.collect()

[1, 2, 3, 4]

In [24]:
# reduceByKey(func)
rdd = sc.parallelize([("A",1), ("B",2), ("A",3)])
rdd_reduce = rdd.reduceByKey(lambda x, y: x + y)
rdd_reduce.collect()

[('A', 4), ('B', 2)]

In [26]:
# grouByKey()
rdd = sc.parallelize([("A", 1), ("B", 2), ("A", 3)])
rdd_groupby = rdd.groupByKey().mapValues(list)
rdd_groupby.collect()

[('A', [1, 3]), ('B', [2])]

In [28]:
# aggregateByKey(zeroValue, seqFunc, combFunc)
rdd = sc.parallelize([("A",10), ("B",20), ("A",30)])
zero_val = (0,0)
seqFunc = lambda x, y: (x[0] + y, x[1] + 1)
combFunc = lambda x, y: (x[0] + y[0], x[1] + y[1])
rdd_avg = rdd.aggregateByKey(zero_val, seqFunc, combFunc)
rdd_avg.collect()

[('A', (40, 2)), ('B', (20, 1))]

In [32]:
# join(otherDataset)
rdd1 = sc.parallelize([("A", 1), ("B", 2)])
rdd2 = sc.parallelize([("A", "X"), ("C", "Y")])
rdd_joined = rdd1.join(rdd2)  # Resultado: [('A', (1, 'X'))]
rdd_joined.collect()

[('A', (1, 'X'))]

In [33]:
# distinct()
rdd = sc.parallelize([1, 2, 2, 3, 1])
rdd_distinct = rdd.distinct()  # Resultado: [1, 2, 3]
rdd_distinct.collect()

[2, 1, 3]

In [34]:
# repartition(numPartition)
rdd = sc.parallelize([1, 2, 3, 4], 2) # Inicial: 2 particiones
rdd_repartition = rdd.repartition(4)  # Ahora: 4 particiones (Dispara Shuffle)
rdd_repartition.getNumPartitions()

4

In [35]:
# sortBy(keyfunc, ascending=True)
rdd = sc.parallelize([3, 1, 4, 2])
rdd_sorted = rdd.sortBy(lambda x: x)  # Resultado: [1, 2, 3, 4]
rdd_sorted.collect()

[1, 2, 3, 4]

## Ejercicios

In [22]:
# Cree un RDD llamado lenguajes que contenga los siguiente lenguajes de programación: Python, R, C, Scala, Ruby y SQL
rdd_lenguajes = sc.parallelize(["Python", "R", "C", "Scala", "Ruby", "SQL"])
rdd_lenguajes.collect()

['Python', 'R', 'C', 'Scala', 'Ruby', 'SQL']

In [4]:
# Obtenga un nuevo RDD a partir del RDD lenguajes donde todos los lenguajs de programacion estén en mayúsculas
rdd_mayusculas = rdd_lenguajes.map(lambda x: x.upper())
rdd_mayusculas.collect()

['PYTHON', 'R', 'C', 'SCALA', 'RUBY', 'SQL']

In [6]:
# Obtenga un nuevo RDD a partir del RDD lenguajes donde todos los lenguajes de programación estén en minúsculas
rdd_minusculas = rdd_lenguajes.map(lambda x: x.lower())
rdd_minusculas.collect()

['python', 'r', 'c', 'scala', 'ruby', 'sql']

In [8]:
# Cree un nuevo RDD que solo contenga aquellos lenguajes de programación que comiencen con la letra R
rdd_startr = rdd_lenguajes.filter(lambda x: x.startswith("R"))
rdd_startr.collect()

['R', 'Ruby']

In [10]:
# Crea un RDD llamado pares que contenga los números pares existentes en el intervalo [20,30]
rdd_numeros = sc.parallelize(range(20,31))
rdd_pares = rdd_numeros.filter(lambda x: x % 2 == 0)
rdd_pares.collect()

[20, 22, 24, 26, 28, 30]

In [15]:
# Cree el RDD llamado sqrt, este debe contener la raíz cuadrada de los elementos que componene el RDD pares
from math import sqrt
rdd_sqrt = rdd_pares.map(lambda x: sqrt(x))
rdd_sqrt.collect()

[4.47213595499958,
 4.69041575982343,
 4.898979485566356,
 5.0990195135927845,
 5.291502622129181,
 5.477225575051661]

In [17]:
# Obtenga una lista compuesta por lo número pares en el intervalo [20,30] y sus respectivas raíces cuadradas.
rdd_sqrt2 = rdd_pares.flatMap(lambda x: (x, sqrt(x)))
rdd_sqrt2.collect()

[20,
 4.47213595499958,
 22,
 4.69041575982343,
 24,
 4.898979485566356,
 26,
 5.0990195135927845,
 28,
 5.291502622129181,
 30,
 5.477225575051661]

In [18]:
# Eleve el número de paticiones del RDD sqrt a 20
rdd_partition20 = rdd_sqrt2.repartition(20)
rdd_partition20.getNumPartitions()

20